# SynnoDB: From DuckDB to Bespoke in One Import

SynnoDB is a drop-in replacement for DuckDB that transparently accelerates your SQL queries
with auto-generated bespoke C++ engines - while falling back to DuckDB for everything else.
No schema changes. No query rewrites. One import.

This notebook builds a **Simple Bespoke Engine** - *few queries, low optimization*. It takes a
handful of query templates (**TPC-H Q1-Q5**), generates one naive-but-parallel C++ engine for
them, and drops it in under DuckDB - the quickest path from SQL to a working bespoke engine.
The complementary **Full Bespoke Engine** - *many queries, high optimization* - pushes such an
engine through extra tuning passes for peak performance; that path is out of scope here.

At a high level:

1. **Baseline** - run Q1-Q5 on vanilla DuckDB at SF5, 10 parameter instantiations each.
2. **Generate** - hand SynnoDB a live DuckDB connection plus a `queries.json` and let it write
   the engine.
3. **Drop in** - replace one import, re-run the identical queries, compare the numbers.

It is **self-contained**: it builds a single TPC-H DuckDB with DuckDB's own `dbgen` (no manual
download, no external tooling), and SynnoDB **derives its own cheap correctness rungs by
FK-preserving downscaling** of that one connection - you no longer supply pre-scaled data tiers.

### Prerequisites

**Install SynnoDB** - it lives on [PyPI](https://pypi.org/project/synnodb/), so a single
`pip install` pulls in every Python dependency. This demo *generates* an engine, so add the
`factory` extra:

```bash
pip install "synnodb[factory]"
```

(Working from a source checkout? `pip install -e ".[factory]"` from the repo root does the same.)

## Setup

One knob - the **data root**. Set `SYNNO_DATA_DIR` (env or `.env`) to where your data should
live; unset, it defaults to a project-local `.synno_data/`. It does **not** need to exist yet:
the next cell builds a `tpch.duckdb` inside it on first run.

In [ ]:
import logging
import os, json, time, statistics
from pathlib import Path

from dotenv import load_dotenv

from synnodb.utils.path_utils import repo_root
from synnodb.observability.logging.logger import setup_logging
setup_logging(logging.INFO)
load_dotenv()  # let SYNNO_DATA_DIR / SYNNO_ENGINES_DIR / SYNNO_WORKSPACE come from a .env

# The data root holds everything: the source DuckDB, engines, workspace. Honor SYNNO_DATA_DIR if
# set, else default to a project-local .synno_data/. It need not exist yet - the next cell builds
# the TPC-H DuckDB into it.
DATA_ROOT = Path(os.environ.get("SYNNO_DATA_DIR") or repo_root() / ".synno_data")
ENGINES_DIR = DATA_ROOT / "engines"
TPCH_DB = DATA_ROOT / "tpch.duckdb"   # the single source of truth - one live DuckDB

SF = 5                                # scale factor of the *source* DuckDB we build once
# SynnoDB derives these fast-check rungs itself, by FK-preserving downscaling of the source - no
# pre-scaled tiers. Each is a sampling ratio of the largest table; joins stay non-vacuous so the
# cheap correctness check still catches real bugs.
DOWNSCALE_TIERS = (0.02, 0.1)

TABLES = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]

MODEL = os.environ.get(
    "SYNNO_MODEL", "anthropic/claude-sonnet-5"
)  # e.g. "anthropic/claude-sonnet-4-6", "gpt-5.4", "openrouter/z-ai/glm-5.2"
MODEL_EXTRA_BODY = json.loads(os.environ.get("SYNNO_MODEL_EXTRA_BODY", "null"))
QUERIES_JSON = Path("queries.json")  # lives next to this notebook

print("Data root   :", DATA_ROOT)
print("Source DB   :", TPCH_DB)
print("Engines dir :", ENGINES_DIR)
print("Model       :", MODEL)

### Build the source DuckDB (first run only)

To keep the notebook self-contained we materialize the TPC-H tables with DuckDB's built-in
`dbgen` into a single `tpch.duckdb` - no external download, no pre-scaled files. This one DuckDB
**is** the dataset: SynnoDB reads it, benchmarks against it at full scale, and derives the cheap
correctness rungs from it automatically (next section).

**SF5 is a couple of GB and takes a little while to build**; make sure you have the disk.
Generation caps DuckDB's memory and spills to a temp directory so it does not OOM. The step is
idempotent - a `tpch.duckdb` already holding every table is reused.

In [ ]:
from synnodb.workloads.dataset.gen_tpc_h_data import ensure_tpch_duckdb

ensure_tpch_duckdb(TPCH_DB, SF)

---
## Step 1 - Workload Registration

We describe the workload once - its **queries** - and hand SynnoDB the **live DuckDB**. From that
one connection it reads the schema, benchmarks Q1-Q5 at full scale, and **derives the cheap
correctness rungs by FK-preserving downscaling** (no pre-scaled tiers). We also run Q1-Q5 on
vanilla DuckDB at SF5 (10 instantiations per query) as the baseline; those exact SQL strings are
reused for the SynnoDB comparison, so it stays apples-to-apples.

### Register the workload from the DuckDB connection

The workload is described by a single self-describing JSON file. Each entry carries its SQL
template **and** a typed **spec** for each `[PLACEHOLDER]` slot - declaring its value *space*,
which is sampled at run time. A scalar placeholder is an `int`/`float`/`date`/`categorical`
spec; correlated or distinct placeholders share a `param_groups` spec:

```json
"6": {
  "sql": "... l_discount between [DISCOUNT] - 0.01 ... l_quantity < [QUANTITY] ...",
  "params": {
    "DATE":     { "type": "date",  "min": "1993-01-01", "max": "1997-01-01" },
    "DISCOUNT": { "type": "float", "min": 0.02, "max": 0.09, "step": 0.01 },
    "QUANTITY": { "type": "int",   "min": 24, "max": 25 }
  }
},
"7": {
  "sql": "... n1.n_name = '[NATION1]' ... n2.n_name = '[NATION2]' ...",
  "param_groups": [
    { "type": "sample", "placeholders": ["NATION1", "NATION2"], "domain": ["GERMANY", "CHINA", ...], "distinct": true }
  ]
}
```

`db.sync_from_duckdb(...)` reads the queries, **infers the join graph straight from their
JOINs**, and downscales the source **referentially** - anchored on the largest table and
following the join graph - so a 2% rung still joins to real rows instead of collapsing to
near-empty. The full dataset is the benchmark tier, and nothing is ever copied back into your
DuckDB. Each placeholder's typed spec is what a BI dashboard renders as a slider (`int`/`float`),
a dropdown (`categorical`), or a date-picker (`date`).

In [ ]:
from synnodb import SynnoDB

# Constructing the driver spawns an in-process live-UI dashboard and prints its URL
# (e.g. http://localhost:8765). Open it in a browser to watch generation unfold in real time -
# input tokens, code size, per-query speedups, cost/runtime, and an activity log. The dashboard
# stays up for the lifetime of this kernel; the URL is also available as `db.dashboard_url`.
db = SynnoDB(
    model=MODEL,
    model_extra_body=MODEL_EXTRA_BODY,
    db_storage="in_memory",
    queries="1-5",
    data_dir=DATA_ROOT,
)

# Hand the live DuckDB to SynnoDB. It reads the schema + queries, materializes the benchmark tier
# and the FK-preserving fast-check rungs, and registers the workload - all from this one
# connection, which is only ever read (nothing is written back).
spec = db.sync_from_duckdb(
    TPCH_DB,  # a path (opened read-only) or a live duckdb.DuckDBPyConnection
    name="tpch_byo",
    queries_json=QUERIES_JSON,
    downscale_tiers=DOWNSCALE_TIERS,
    schema_example_table="lineitem",
)

print("Workload :", spec.name)
print("Tables   :", spec.tables)
print("Queries  :", spec.all_query_ids)
print("Tiers    :", spec.exhaustive_sfs, "(benchmark:", spec.benchmark_sf, ")")

Here is what the queries look like - SQL templates with `[PLACEHOLDER]` slots, plus the typed
specs that define each slot's value space:

In [ ]:
queries = json.loads(QUERIES_JSON.read_text())
for qid, entry in queries.items():
    print(f"=== Q{qid} ===")
    print(entry["sql"][:240], "...")
    print("params      :", entry.get("params", {}))
    print("param_groups:", entry.get("param_groups", []))
    print()

### Draw parameter instantiations

`query_gen_factory` fills the templates by sampling each placeholder's spec. We draw with a
fixed seed so the instantiations are **identical** for the DuckDB and SynnoDB runs.

In [ ]:
N_REPS = 10
rng = random.Random(42)
gen = spec.query_gen_factory(None)

# gen(query_name, rng) -> (query_name, sql_with_values, params_dict)
instantiations = {}
for qid in spec.all_query_ids:
    instantiations[qid] = [gen(f"Q{qid}", rng) for _ in range(N_REPS)]

print(f"Drew {N_REPS} instantiations per query.")
for qid, insts in instantiations.items():
    sample_params = [i[2] for i in insts[:2]]
    print(f"  Q{qid}: {sample_params} ...")

---
## Step 2 - Generate the Simple Bespoke Engine

You already handed SynnoDB the queries and the live DuckDB. It:

1. **Creates a storage plan** - decides how each query's columns are laid out in memory.
2. **Implements the engine** - writes a naive multi-threaded C++ engine, compiles it, validates
   every output against DuckDB on the cheap downscaled rungs first (then the full tier), and
   **auto-publishes** the binary into `ENGINES_DIR`.

That is the whole Simple Bespoke Engine: correct and parallel, but only lightly tuned - and
already fast enough to beat DuckDB on these queries. It is a one-time cost; once published the
engine is discovered automatically across sessions.

### Storage plan

In [ ]:
# Expected cost: ~$0.1
# Expected time: ~2 mins / 6 turns (depending on model speed)

plan = db.createStoragePlan()

print(plan.text[:600], "...")

### Base implementation

We feed the plan **content** straight in via `storage_plan=plan.text`, so this step needs
no W&B. If you instead chain off a logged storage-plan run, pass its run id with
`db.createBaseImpl(storage_plan_wandb_id=plan.run_id)`. Provide exactly one of the two.

This is the one generation step this notebook runs, and it produces the complete Simple Bespoke
Engine: a **naive multi-threaded** C++ engine - correct and parallel, but not yet
performance-tuned. For **TPC-H Q1-Q5** it costs about **$6** and finishes in **under 500
turns**, and already delivers **2-10x** speedups over DuckDB.

In [ ]:
# Expected cost: ~$6
# Expected time: ~1 hrs / 500 turns (depending on model speed)
impl = db.createBaseImpl(storage_plan=plan.text)

print("Workspace :", impl.workspace)
print("Files     :", sorted(impl.files))
print()
print(f"Engine published to: {ENGINES_DIR}")

---
# Step 3 - Benchmark DuckDB
### Run queries on DuckDB as comparison baseline

In [ ]:
import duckdb
import tempfile
from tqdm import tqdm

# Threads both engines run at, so DuckDB and SynnoDB are compared at the same parallelism.
NUM_THREADS = 1  # 1 for demo, else os.cpu_count() for benchmark
assert NUM_THREADS is not None, "os.cpu_count() returned None"

print(f"Measuring DuckDB performance with {NUM_THREADS} threads...")
duck = duckdb.connect(":memory:", config={"threads": NUM_THREADS})
duck.execute("PRAGMA disable_progress_bar")
duck.execute("PRAGMA enable_profiling='json'")  # EXPLAIN ANALYZE returns its profile as JSON
# enable_profiling also makes DuckDB dump a JSON profile to the console after *every* statement
# (the CREATE TABLEs below, every EXPLAIN ANALYZE) - a wall of text in the notebook. Send those
# automatic dumps to a throwaway file; the EXPLAIN ANALYZE result set still carries the profile
# analyze_ms() reads, so the timings are unaffected.
duck.execute(f"PRAGMA profiling_output='{Path(tempfile.gettempdir()) / 'duckdb_profile.json'}'")
# Load the full-scale tables into memory from the source DuckDB (CREATE TABLE, not a view over
# the file), so the measured query time is in-memory execution - the same basis SynnoDB's engine
# serves at - not a buffer-managed scan off disk on every run.
duck.execute(f"ATTACH '{TPCH_DB.as_posix()}' AS src (READ_ONLY)")
for t in TABLES:
    duck.execute(f"CREATE TABLE {t} AS SELECT * FROM src.{t}")
duck.execute("DETACH src")


def analyze_ms(con, sql):
    """DuckDB's own execution latency for `sql`, via EXPLAIN ANALYZE - the server-side query
    time, excluding the Python call and result-fetch overhead a wall-clock timer would include.
    The json profile (whose `latency` is in seconds) is the second column of the result row."""
    profile = json.loads(con.execute("EXPLAIN ANALYZE " + sql).fetchone()[1])
    return profile["latency"] * 1_000


baseline_times = {}
for qid, insts in tqdm(instantiations.items(), desc="Measuring DuckDB performance"):
    baseline_times[qid] = [analyze_ms(duck, sql) for _, sql, _ in insts]

duck.close()

total_duck = sum(sum(v) / len(v) for v in baseline_times.values())
print(f"{'Query':<8} {'Avg (ms)':>12} {'Median (ms)':>14}")
print("-" * 38)
for qid in spec.all_query_ids:
    t = baseline_times[qid]
    print(f"Q{qid:<7} {sum(t) / len(t):>12.1f} {statistics.median(t):>14.1f}")
print("-" * 38)
print(f"{'TOTAL':<8} {total_duck:>12.1f}")

---
## Step 4 - Drop In SynnoDB

The only change is **one import line**, a few extra keyword arguments to `connect()`, and you
point it straight at your DuckDB file:

```diff
- import duckdb
+ import synnodb
+ from synnodb.router import RouterMode, RouterPolicy

-  con = duckdb.connect("tpch.duckdb")
+  con = synnodb.connect(
+      "tpch.duckdb",
+      config={"threads": NUM_THREADS},   # same knob DuckDB got - fixes the engine's parallelism too
+      engines=str(ENGINES_DIR),
+      policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
+  )
```

The tables already live in the DuckDB file, so there is nothing to load by hand - every other
line (the `execute()` calls, `fetchall()`) is identical.

### Open the drop-in connection

In [ ]:
import synnodb
from synnodb.router import RouterMode, RouterPolicy

# Open the drop-in connection straight on the source DuckDB - its tables are already present, so
# (unlike the DuckDB baseline) there is nothing to CREATE by hand. The bespoke engine ingests
# those tables into its own in-memory snapshot when we warm it below.
con = synnodb.connect(
    str(TPCH_DB),
    config={"threads": NUM_THREADS},  # same thread budget as the DuckDB baseline above
    engines=str(ENGINES_DIR),
    policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
)

con.refresh_engines()
n = con.router_stats()["registry"]["templates"]
print(f"Discovered {n} engine template(s) under {ENGINES_DIR}")

# Load each engine's data now - start its process, ingest the live DuckDB tables, build its
# in-memory database - so this one-time cost is paid here as an explicit step instead of landing
# on the first query.
n_warmed = con.synno_ingest_data()
print(f"Preloaded {n_warmed} engine(s) - first query is served warm")

### Run the same queries with the same parameter values

We iterate over `instantiations` - the exact SQL strings from Step 1. For each we record the
engine's **own execution latency** (`engine_ms`, as measured by the router), the same
server-side basis the DuckDB baseline uses via `EXPLAIN ANALYZE` - so the two columns compare
like with like rather than one wall-clock against another.

In [ ]:
synno_times = {}
routed_to = {}  # per query: "SynnoDB" if the bespoke engine served it, else "DuckDB"
for qid, insts in tqdm(instantiations.items(), desc="Running queries through SynnoDB router:"):
    times = []
    backends = []
    for _, sql, _ in insts:
        con.execute(sql).fetchall()
        # `engine_ms` is present when the bespoke engine served the query;
        # a query that falls back to DuckDB exposes `duckdb_ms` instead.
        last = con._last
        served_bespoke = "engine_ms" in last
        times.append(last["engine_ms"] if served_bespoke else last["duckdb_ms"])
        backends.append(served_bespoke)
    synno_times[qid] = times
    routed_to[qid] = "SynnoDB" if all(backends) else "DuckDB"

### Speedup

In [ ]:
total_synno = sum(sum(v)/len(v) for v in synno_times.values())

print(f"{'Query':<8} {'Routing':>8} {'DuckDB (ms)':>12} {'SynnoDB (ms)':>14} {'Speedup':>9}")
print("-" * 55)
for qid in spec.all_query_ids:
    avg_d = sum(baseline_times[qid])/len(baseline_times[qid])
    avg_s = sum(synno_times[qid])/len(synno_times[qid])
    speedup = avg_d / avg_s if avg_s > 0 else float("inf")
    # ⚡ marks queries the bespoke SynnoDB engine served; the rest fell back to DuckDB.
    routing = routed_to[qid]
    mark = " ⚡" if routing == "SynnoDB" else ""
    print(f"Q{qid:<7} {routing:>8} {avg_d:>12.1f} {avg_s:>14.1f} {speedup:>8.2f}x{mark}")
print("-" * 55)
overall = total_duck / total_synno if total_synno > 0 else float("inf")
print(f"{'TOTAL':<8} {'':>8} {total_duck:>12.1f} {total_synno:>14.1f} {overall:>8.2f}x")

### Correctness guarantee

Every routed result was compared against DuckDB (`cross_check_rate=1.0`).
The mismatch count must be 0.

In [ ]:
stats = con.router_stats()["session"]
print(f"Routed:         {stats['routed']}")
print(f"Cross-checked:  {stats['cross_checked']}")
print(f"Mismatches:     {stats['cross_check_mismatch']}")
assert stats["cross_check_mismatch"] == 0, "result divergence detected!"
print("\nAll results match DuckDB exactly.")

### Q1 result (with timing footer)

In [ ]:
_, q1_sql, _ = instantiations["1"][0]
con.execute(q1_sql)
con.show()
con.close()

---
## Bonus - Define Your Own Conversation

Every built-in stage above is an ordinary `ConversationPlan` - and you can assemble
your own from the same primitives. A plan names the run (for logging and caching),
states *what the prepared workspace must provide* (`PrepareFeatures`), and supplies a
`stages` callable that turns a `ConvContext` into a flat list of stage items:

- `PromptStage` - one declarative LLM task with measurement/revert flags; its prompt
  callback receives the freshly measured runtime (and trace data, if requested).
- `PerQueryLoop` - one conversation branch per query, stages executed ring by ring.
- markers (`Compact`, `Benchmark`, `ValidateOn`, ...) and checks (`AssertCorrect`).

`db.run_synthesis(plan, start=...)` is the single entry point every stage goes
through; `start` chains off any earlier artifact (here: the base implementation).
The returned artifact carries the final snapshot hash and the workspace's prepare
record, so it chains onwards (e.g. into `db.checkSfCorrectness(result, target_sf=100)`).


In [ ]:
from synnodb import (
    AssertCorrect, Benchmark, Compact, ConversationPlan, ConvContext,
    PerQueryLoop, PrepareFeatures, PromptStage,
)

def my_stages(ctx: ConvContext):
    return [
        AssertCorrect(),
        PromptStage(
            descriptor="inspect hot loops",
            get_prompt=lambda _exec_settings, _rt: (
                f"Profile {ctx.filenames.query_impl_path} and summarize the hot loops."),
            measure_performance_after_stage=False,
            auto_revert_on_regression=False,
        ),
        Compact(),
        PerQueryLoop(lambda qid, ctx: [
            PromptStage(
                descriptor=f"tune {qid}",
                get_prompt_with_tracing=lambda _exec_settings, rt, trace: (
                    f"Query {qid} currently runs in {rt:.0f} ms.\n"
                    f"Trace:\n{trace}\nOptimize it."),
                max_turns=125,
                # defaults: measure after stage, auto-revert on regression
            ),
        ]),
        Benchmark(),
    ]

tuning_plan = ConversationPlan(
    name="myTuningPass",                    # run identity: naming, logging, caching
    prepare=PrepareFeatures(tracing=True),  # the workspace needs tracing instrumentation
    stages=my_stages,
)

# Uncomment to run the custom pass on top of the base implementation:
# tuned = db.run_synthesis(tuning_plan, start=impl)


---
## Where to go next

This notebook built the **Simple Bespoke Engine** - few queries, lightly tuned. The natural
next step is the **Full Bespoke Engine**: a larger query set driven through extra tuning passes
for peak performance. That path is covered separately and is out of scope for this demo.

You can already validate the engine you just built at a larger scale factor, restored straight
from its local git snapshot (`impl.snapshot_hash`) - no W&B required:

```python
rep = db.checkSfCorrectness(source=impl, target_sf=50)   # correctness at a larger scale factor
```

> **Want W&B logging?** Pass `wandb_project="..."` (and/or `wandb_entity="..."`) to
> `SynnoDB(...)`. W&B is off unless one of them is set - nothing logs in, initializes, or
> requires credentials otherwise.